### 解决方案概述


我们的解决方案整体流程如下图所示。

![
alt](https://bohrium.oss-cn-zhangjiakou.aliyuncs.com/article/27254/ddeaac53cdf54dbabcb634f54656af77/9d89f692-dd1d-4c2f-91b7-e415b91323e7.jpeg)

本方案的训练数据全部来自于比赛所提供的数据，并未使用外部数据。

本方案主要分为pdf解析、多模态数据处理、模型训练与预测3个部分。由于3个部分使用的工具不同，且工具之间的依赖有冲突，所以本方案共使用了3套conda环境，大部分代码都是py文件和sh文件，并利用比赛提交的notebook作为shell环境来运行。

---


我们针对task1～task7中的不同模态数据，分别进行了算法设计。

task1为知识问答题，task2～task7则全部需要基于pdf内容做问答。task2～task7的pdf解析，我们使用的是上海人工智能实验室的[MinerU](https://github.com/opendatalab/MinerU)，它可以将pdf解析为markdown文件，将表格、图片等提取保存到本地，并在文中替换为保存的本地路径。

对于RAG过程，由于MinerU解析的输出为markdown文件，所以我们首先利用langchain对md文件进行了切分，然后利用bge-m3筛选出与问题相似度最高的10个块，最后利用bge-reranker-v2模型进行召回。


**task1**：由于task1的训练数据相比于其他任务较多，出于稳定性考虑，我们随机将task1的训练数据分为了3份，分别对yi1.5-34b-chat进行qlora微调，最终的结果由3个模型投票得到。

**task2**：由于task2涉及到了表格模态，所以我们利用MinerU对pdf进行解析，将所有的表格都保存为图片，然后利用MiniCPM-V-2.6对表格图片进行提取，将提取的内容替换到markdown的相应位置，最后对markdown进行RAG，检索出与问题相关的内容添加到问题后面。以上操作训练过程与推理过程是相同的，训练过程通过上述流程构建训练数据，然后对yi1.5-34b-chat进行qlora微调，推理过程则利用微调好的模型进行预测。

**task3**：此任务是基于pdf中的图表进行问答，并且给定了图片所在的pdf页码。我们将图片所在页整页保存为了一张图片，并利用提供的训练集对MiniCPM-V-2.6进行了微调，最终通过微调后的模型进行推理。

**task4**：此任务涉及分子式模态，并且基本都是专利文档。我们前期利用MinerU对pdf进行解析后，利用[MolScribe](https://github.com/thomas0809/MolScribe)将文档中的分子式图片转换为smiles文本，然后进行RAG检索，最后利用微调好的yi1.5-34b-chat进行推理。但是后续随着方案的逐渐完善，出现了运行超时的问题，所以出于时间限制的考虑，我们暂时放弃了此任务
。期待其他选手对于此任务的解决方案。

**task5**：此任务涉及化学反应式模态，我们前期利用MinerU对pdf进行解析后，利用[RxnScribe](https://github.com/thomas0809/RxnScribe)将文档中的反应式图片转换为smiles文本，然后进行RAG检索，最后利用微调好的yi1.5-34b-chat进行推理。

**task6**：此任务只涉及表格模态。我们利用MinerU对pdf进行解析，将所有的表格都保存为图片，然后利用MiniCPM-V-2.6对表格图片进行提取，将提取的内容替换到markdown的相应位置，然后进行RAG，检索出与问题最相关的表格图片，最后利用微调好的MiniCPM-V-2.6模型进行推理。

**task7**：此任务只涉及文本模态。利用MinerU对pdf进行解析，然后对markdown进行RAG，检索出与问题相关的内容添加到问题后面，最后利用训练好的yi1.5-34b-chat模型
进行推理。

### pdf内容解析

#### mineru环境创建与配置

In [1]:
!conda create -n mineru python=3.10 -y

In [2]:
!conda run -n mineru pip install --no-index --find-links=file:/bohr/sci-pkgs-w6lh/v1/pkgs -r /bohr/sci-re-0lem/v1/pkgs_mineru.txt

In [3]:
!cp /bohr/sci-files-t0za/v31/magic-pdf.template.json ~/magic-pdf.json

In [4]:
!apt-get update
!apt-get install libgl1-mesa-glx -y

In [5]:
!mkdir test_mid_mds
!mkdir task3_imgs
!cp /bohr/sci-files-t0za/v31/test_pdf2md.py .
!cp /bohr/sci-files-t0za/v31/test_task2_img2table.py .
!cp /bohr/sci-files-t0za/v31/test_task5_img2react.py .
!cp /bohr/sci-files-t0za/v31/test_task6_img2table.py .
!cp /bohr/sci-files-t0za/v31/bulid_test_data.py .
!cp /bohr/sci-files-t0za/v31/task2_infer.py .
!cp /bohr/sci-files-t0za/v31/task3_infer.py .
!cp /bohr/sci-files-t0za/v31/task5_infer.py .
!cp /bohr/sci-files-t0za/v31/task6_infer.py .
!cp /bohr/sci-files-t0za/v31/task7_infer.py .
!cp /bohr/sci-files-t0za/v31/task1_infer_1.py .
!cp /bohr/sci-files-t0za/v31/task1_infer_2.py .
!cp /bohr/sci-files-t0za/v31/task1_infer_3.py .

#### 解析所有使用到的测试集pdf

In [6]:
!conda run -n mineru python test_pdf2md.py

### md中的特殊模态内容转化为文本

#### rxn环境创建与配置

In [7]:
!conda create -n rxn python=3.10 -y

In [8]:
!conda run -n rxn pip install --no-index --find-links=file:/bohr/sci-pkgs-w6lh/v1/pkgs -r /bohr/sci-re-0lem/v1/pkgs_rxn.txt

#### vllm推理主环境创建与配置

In [9]:
!pip install --no-index --find-links=file:/bohr/sci-pkgs-w6lh/v1/pkgs -r /bohr/sci-re-0lem/v1/pkgs_vllm.txt

#### 每个任务的多模态内容转化为文本

In [10]:
!python test_task2_img2table.py

In [11]:
!conda run -n rxn python test_task5_img2react.py

In [12]:
!python test_task6_img2table.py

### rag

In [13]:
!python bulid_test_data.py


### infer

In [14]:
!python task1_infer_1.py

In [15]:
!python task1_infer_2.py

In [16]:
!python task1_infer_3.py

In [17]:
!python task2_infer.py

In [18]:
!python task3_infer.py

In [19]:
!python task5_infer.py

In [20]:
!python task6_infer.py

In [21]:
!python task7_infer.py

In [22]:
import jsonlines
import os

In [24]:
task1_answer_1 = []
with jsonlines.open('task1_answer_1.jsonl', 'r') as data:
    for i in data:
        ideal = i['ideal'].strip().lower()[0]
        task1_answer_1.append(ideal)

In [ ]:
task1_answer_2 = []
with jsonlines.open('task1_answer_2.jsonl', 'r') as data:
    for i in data:
        ideal = i['ideal'].strip().lower()[0]
        task1_answer_2.append(ideal)

In [ ]:
task1_answer_3 = []
with jsonlines.open('task1_answer_3.jsonl', 'r') as data:
    for i in data:
        ideal = i['ideal'].strip().lower()[0]
        task1_answer_3.append(ideal)

In [ ]:
from collections import Counter
def most_frequent_char(char_array):
    counter = Counter(char_array)
    most_common_char = counter.most_common(1)[0][0]
    return most_common_char

In [ ]:
task1_answer = []
with jsonlines.open('task1_answer_1.jsonl', 'r') as data:
    for n, i in enumerate(data):
        i['ideal'] = most_frequent_char([task1_answer_1[n], task1_answer_2[n], task1_answer_3[n]])
        task1_answer.append(i)

In [ ]:
task2_answer = []
with jsonlines.open('task2_answer.jsonl', 'r') as data:
    for i in data:
        i['ideal'] = i['ideal'].strip().capitalize()
        task2_answer.append(i)

In [ ]:
task3_answer = []
with jsonlines.open('task3_answer.jsonl', 'r') as data:
    for i in data:
        i['ideal'] = i['ideal'].strip().lower()
        task3_answer.append(i)

In [ ]:
DATA_PATH=os.getenv('DATA_PATH')
if not DATA_PATH:
    DATA_PATH='/bohr/exampleData-pi6b/v6'
    print("Warning: DATA_PATH environment variable is not set. Using default path:", DATA_PATH)
test_input_path = DATA_PATH+'/question.jsonl'

task4_answer = []
with jsonlines.open(test_input_path, 'r') as data:
    for i in data:
        if i['task'] == '4':
            i['ideal'] = ["Yes"]
            task4_answer.append(i)

In [ ]:
task5_answer = []
with jsonlines.open('task5_answer.jsonl', 'r') as data:
    for i in data:
        i['ideal'] = i['ideal'].strip().lower()
        task5_answer.append(i)

In [ ]:
task6_answer = []
with jsonlines.open('task6_answer.jsonl', 'r') as data:
    for i in data:
        i['ideal'] = i['ideal'].strip().lower()
        task6_answer.append(i)

In [ ]:
task7_answer = []
with jsonlines.open('task7_answer.jsonl', 'r') as data:
    for i in data:
        i['ideal'] = i['ideal'].strip().lower()
        task7_answer.append(i)

In [ ]:
result = task1_answer +  task2_answer +  task3_answer +  task4_answer +  task5_answer +  task6_answer +  task7_answer
with jsonlines.open('submission_all.jsonl', 'w') as file:
    file.write_all(result)

In [ ]:
new_data_a = []
new_data_b = []
with jsonlines.open('submission_all.jsonl', 'r') as data:
    for sample in data:
        if sample['type']=='a':
            new_data_a.append(sample)
        if sample['type']=='b':
            new_data_b.append(sample)

In [ ]:
new_data = new_data_a + new_data_b
with jsonlines.open('submission.jsonl', 'w') as file:
    file.write_all(new_data)